# M4 · Prix et moment de vente — entraînement

**Tâche** : prévoir le prix de l'huile d'olive pour les 6 prochains mois et conseiller **vendre maintenant**
ou **stocker** (si le prix prévu, moins le coût de stockage de 0,15 TND/kg/mois, dépasse le prix d'aujourd'hui).

| | |
|---|---|
| Données | Prix mensuel de l'huile d'olive extra vierge (**FMI**, série FRED `POLVOILUSDM`, USD/t, depuis 1992), converti en **TND/kg** avec le taux de change officiel annuel de la **Banque mondiale**. Si l'équipe a saisi les prix au producteur tunisiens du **COI** dans `data/raw/m4_price/coi_tunisia_producer.csv` (≥ 60 mois), ils sont utilisés à la place. |
| Découpage | **chronologique** : train ≤ 2016 · validation 2017–2020 · test 2021 → dernier mois publié |
| Baseline | prix naïf (le prix reste celui du dernier mois) |
| Modèles | **SARIMA** (ordre choisi par AIC) et **LSTM** (ensemble de 5 graines), prévisions glissantes : à chaque mois du test, le modèle ne voit que le passé |
| Métriques | **MAPE** par horizon (1 à 6 mois) et **gain simulé** du conseil vendre / stocker face à « toujours vendre tout de suite » |
| Export | `models/m4/` : `model.json` (ordre SARIMA), `prices.csv` (série), `metadata.json` |

In [ ]:
import json
import platform

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import m4_common as mc

mc.set_seed()
mc.OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.precision", 3)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## 1 · Données

In [ ]:
series, source = mc.build_series()
print("source :", source)
print(f"{len(series)} mois, de {series.index.min():%Y-%m} à {series.index.max():%Y-%m}")
gaps = pd.date_range(series.index.min(), series.index.max(), freq="MS").difference(series.index)
print("mois manquants :", len(gaps))
series.tail(6)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
series["price"].plot(ax=ax, color="tab:green")
ax.set(title="Prix de l'huile d'olive (TND/kg)", ylabel="TND/kg", xlabel="")
for start, label in [("2021-01-01", "test")]:
    ax.axvline(pd.Timestamp(start), color="grey", ls="--")
    ax.text(pd.Timestamp(start), ax.get_ylim()[1] * 0.95, " " + label, color="grey")
fig.tight_layout()
fig.savefig(mc.OUTPUT_DIR / "price_history.png", dpi=110)
plt.show()
series["price"].describe()

**Lecture** : le prix est très volatil. Après une longue période stable autour de 8–12 TND/kg, la sécheresse
espagnole de 2022–2023 a plus que doublé les prix (record début 2024), puis les bonnes récoltes de 2024/2025
les ont fait chuter. Le test (2021 → aujourd'hui) contient donc la hausse **et** la chute : c'est un test exigeant.

## 2 · Prévisions glissantes sur le test

In [ ]:
log_price = np.log(series["price"])
prices = series["price"].to_numpy()
dates = series.index
test_origins = [d for d in dates if d >= pd.Timestamp("2021-01-01") and d < dates[-1]]
pos = {d: i for i, d in enumerate(dates)}

# actual[k] = [prix au mois d'origine, prix 1..6 mois plus tard] (NaN si pas encore observé)
actual = np.full((len(test_origins), mc.HORIZON + 1), np.nan)
for k, d in enumerate(test_origins):
    window = prices[pos[d] : pos[d] + mc.HORIZON + 1]
    actual[k, : len(window)] = window
print(len(test_origins), "dates de prévision dans le test")

### Baseline naïve

In [ ]:
pred_naive = np.array([np.exp(mc.naive_forecast(log_price.iloc[: pos[d] + 1].to_numpy())) for d in test_origins])

### SARIMA (sur le log du prix)

In [ ]:
order, seasonal, aic_table = mc.select_sarima(log_price[log_price.index <= mc.VAL_END])
print("ordre retenu :", order, seasonal)
aic_table.head(5)

In [ ]:
sarima = [mc.sarima_forecast(log_price.iloc[: pos[d] + 1], order, seasonal) for d in test_origins]
pred_sarima = np.exp(np.array([m for m, _, _ in sarima]))
low_sarima = np.exp(np.array([lo for _, lo, _ in sarima]))
high_sarima = np.exp(np.array([hi for _, _, hi in sarima]))
inside = (actual[:, 1:] >= low_sarima) & (actual[:, 1:] <= high_sarima)
coverage = np.nanmean(np.where(np.isnan(actual[:, 1:]), np.nan, inside), axis=0) * 100
print("couverture de l'intervalle à 80 % par horizon :", np.round(coverage).astype(int), "%")

### LSTM (ensemble de 5 graines, sur les rendements logarithmiques)

In [ ]:
train_origin_end = pd.Timestamp(mc.TRAIN_END) - pd.DateOffset(months=mc.HORIZON)   # cibles <= fin du train
val_origin_end = pd.Timestamp(mc.VAL_END) - pd.DateOffset(months=mc.HORIZON)
x_train, y_train, _ = mc.lstm_samples(log_price, "1994-01-01", train_origin_end)
x_val, y_val, _ = mc.lstm_samples(log_price, pd.Timestamp(mc.TRAIN_END) + pd.DateOffset(months=1), val_origin_end)
print("train", x_train.shape, "| val", x_val.shape)
models, scale = mc.train_lstm_ensemble(x_train, y_train, x_val, y_val)
x_test = mc.lstm_inputs(log_price, test_origins)
pred_lstm = actual[:, :1] * np.exp(mc.lstm_ensemble_predict(models, scale, x_test))

## 3 · Résultats (jeu de test)

In [ ]:
preds = {"naïf (baseline)": pred_naive, "SARIMA": pred_sarima, "LSTM": pred_lstm}
mape = pd.DataFrame({name: mc.mape_by_horizon(actual[:, 1:], p) for name, p in preds.items()},
                    index=pd.RangeIndex(1, mc.HORIZON + 1, name="horizon (mois)")).T
mape["moyenne"] = mape.mean(axis=1)
mape.to_csv(mc.OUTPUT_DIR / "mape.csv")
mape.round(2)

In [ ]:
ax = mape.drop(columns="moyenne").T.plot(marker="o", figsize=(7, 4), title="MAPE du prix selon l'horizon (test)")
ax.set_ylabel("MAPE (%)")
plt.savefig(mc.OUTPUT_DIR / "mape_by_horizon.png", dpi=110)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(dates[dates >= "2020-01-01"], series["price"][dates >= "2020-01-01"], "k-", label="observé")
h = 3
target_dates = [d + pd.DateOffset(months=h) for d in test_origins]
ax.plot(target_dates, pred_sarima[:, h - 1], "--", label=f"SARIMA (prévu {h} mois avant)")
ax.fill_between(target_dates, low_sarima[:, h - 1], high_sarima[:, h - 1], alpha=0.2, label="SARIMA 80 %")
ax.plot(target_dates, pred_lstm[:, h - 1], ":", label=f"LSTM (prévu {h} mois avant)")
ax.set(title=f"Prévisions à {h} mois sur le test", ylabel="TND/kg")
ax.legend()
fig.tight_layout()
fig.savefig(mc.OUTPUT_DIR / "forecasts_h3.png", dpi=110)
plt.show()

## 4 · Le conseil vendre / stocker rapporte-t-il ?

Pour chaque mois (dont les 6 mois suivants sont connus) : le modèle conseille d'attendre `k` mois si le prix
prévu dans `k` mois, moins `0,15 × k` TND/kg de stockage, dépasse le prix d'aujourd'hui **d'une marge de
prudence** (en % du prix du jour). On vend alors au **vrai** prix de ce mois-là, moins le stockage. Comparaison
avec « toujours vendre tout de suite » et avec un oracle qui connaîtrait l'avenir (gain maximum possible).

La marge est choisie sur la **validation (2017–2020)**, puis appliquée telle quelle au test.

In [ ]:
val_origins = [d for d in dates if pd.Timestamp("2017-01-01") <= d <= pd.Timestamp(mc.VAL_END) - pd.DateOffset(months=mc.HORIZON)]
actual_val = np.array([prices[pos[d] : pos[d] + mc.HORIZON + 1] for d in val_origins])
pred_val = np.exp(np.array([mc.sarima_forecast(log_price.iloc[: pos[d] + 1], order, seasonal)[0] for d in val_origins]))

margins = [0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.15, 0.20]
val_table = pd.DataFrame({f"{m:.0%}": mc.strategy_summary(mc.simulate_strategy(actual_val, pred_val, m))
                          for m in margins}).T
best_gain = val_table["gain moyen (TND/kg)"].max()
min_gain = max(m for m in margins if val_table.loc[f"{m:.0%}", "gain moyen (TND/kg)"] >= best_gain - 1e-9)
print(f"marge retenue sur la validation : {min_gain:.0%}")
val_table.round(3)

In [ ]:
rows = {
    "toujours vendre (référence)": mc.strategy_summary(mc.simulate_strategy(actual, pred_naive)),
    "LSTM": mc.strategy_summary(mc.simulate_strategy(actual, pred_lstm, min_gain)),
    "SARIMA sans marge": mc.strategy_summary(mc.simulate_strategy(actual, pred_sarima, 0.0)),
    f"SARIMA marge {min_gain:.0%} (servi)": mc.strategy_summary(mc.simulate_strategy(actual, pred_sarima, min_gain)),
}
strategy = pd.DataFrame(rows).T
strategy.to_csv(mc.OUTPUT_DIR / "strategy.csv")
strategy.round(3)

**Lecture** : sans marge, SARIMA conseille souvent de stocker et a raison la plupart du temps, mais les rares erreurs
coûtent cher (chute de 2024–2025) : le gain moyen peut devenir négatif. La marge de prudence réduit le nombre de
conseils « stocker » aux cas où la hausse prévue est nette. L'oracle montre le gain maximum théorique : prévoir le
prix de l'huile reste difficile, ce que le rapport doit dire clairement.

## 5 · Choix du modèle et export

Le service sert **SARIMA** si sa MAPE moyenne est au moins aussi bonne que celle du LSTM : il donne en plus un
intervalle de confiance (la bande `low` / `high` du contrat) et se réajuste en une seconde sur toute la série au
démarrage du service. Les prévisions mensuelles sont interpolées par semaine pour l'app (`horizonWeeks`).

In [ ]:
best = mape["moyenne"].drop("naïf (baseline)").idxmin()
print("meilleur modèle (MAPE moyenne) :", best)
print(mape["moyenne"].round(2).to_dict())

import statsmodels

model_json = {
    "type": "sarima",
    "order": list(order),
    "seasonal_order": list(seasonal),
    "transform": "log",
    "interval": 0.8,
    "storage_cost_tnd_kg_month": mc.STORAGE_COST,
    "min_gain_share": min_gain,
    "source": source,
    "series_end": f"{series.index.max():%Y-%m}",
}
prices_csv = series[["price"]].rename(columns={"price": "price_tnd_kg"})
prices_csv.index.name = "date"
metadata = {
    "served_model": "SARIMA",
    "best_mean_mape_on_test": best,
    "data": {"source": source, "period": [f"{series.index.min():%Y-%m}", f"{series.index.max():%Y-%m}"],
             "split": {"train": ["1992-01", mc.TRAIN_END[:7]], "val": ["2017-01", mc.VAL_END[:7]],
                       "test": [f"{test_origins[0]:%Y-%m}", f"{test_origins[-1]:%Y-%m}"]}},
    "sarima": {"order": list(order), "seasonal_order": list(seasonal),
               "interval_coverage_80_by_horizon": [round(float(c), 1) for c in coverage]},
    "decision_margin": {"min_gain_share": min_gain, "validation": json.loads(val_table.round(4).to_json(orient="index"))},
    "mape_test_percent": json.loads(mape.round(3).to_json(orient="index")),
    "strategy_test": json.loads(strategy.round(4).to_json(orient="index")),
    "versions": {"python": platform.python_version(), "statsmodels": statsmodels.__version__, "numpy": np.__version__},
}
mc.export({"model.json": model_json, "prices.csv": prices_csv}, metadata)